# Scryfall Tags: Multi-Label Classification  
__Objective:__ Since the scryfall tags are in essence a collection of multiple labels for each card, this problem is at it's core a mutli-label classification task.

## Packages and Data

In [1]:
# packages

## connect project directory
import sys
from pathlib import Path
dir = str(Path(Path.cwd()).parents[0])
if dir not in sys.path:
    sys.path.append(dir)

## load from project directory
from src.data_gathering.scryfall_dataset import ScryfallDataset

In [2]:
# params

## data gathering
from src.config import BUILD_DATASET, TASK, DATASET_SIZE_N, TEST_SIZE_N
from src.config import MAX_INPUT_LENGTH

## modeling
from src.config import MODEL_NAME
from src.config import BATCH_SIZE, LEARNING_RATE, WEIGHT_DECAY, NUM_EPOCHS

In [3]:
# get data
sf = ScryfallDataset()

## build dataset as needed
if BUILD_DATASET:
    sf.build_dataset(
        task = TASK,
        tag_path = '../reports/scryfall_tags.json',
        train_size_pct = 0.8,
        truncate_dataset = DATASET_SIZE_N,
        test_size_n = TEST_SIZE_N
    )

## load dataset
sf.load_hf_dataset(
    train_path = f'../data/scryfall_{TASK}_train.json',
    val_path = f'../data/scryfall_{TASK}_val.json',
    test_path = f'../data/scryfall_{TASK}_test.json'
)

Scryfall Tag Question Answering Dataset Built
	Train Records = 349
	Validation Records = 88
	Test Records = 10
	Records saved to...
		../data/scryfall_multi_label_classification_train.json
		../data/scryfall_multi_label_classification_val.json
		../data/scryfall_multi_label_classification_test.json
	NOTE: This method does not create the huggingface dataset object. Run load_dataset() for that.
Scryfall Tag Multi Label Classification Dataset Loaded
	Train Records = 349
	Val Records = 88
	Test Records = 10


In [4]:
sf.dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'document', 'tags'],
        num_rows: 349
    })
    val: Dataset({
        features: ['id', 'document', 'tags'],
        num_rows: 88
    })
    test: Dataset({
        features: ['id', 'document', 'tags'],
        num_rows: 10
    })
})

## Modeling

In [5]:
x = sf.dataset['train'][0]
x

{'id': 271,
 'document': "\n        Eye of the Storm\n        Mana Cost = {5}{U}{U}\nMana Value = 7.0\n\n        Type Line = Enchantment\n\n        Rules Text = Whenever a player casts an instant or sorcery card, exile it. Then that player copies each instant or sorcery card exiled with this enchantment. For each copy, the player may cast the copy without paying its mana cost.\n \n        \n        \n        Color Identity = ['U']\n\n        Rarity = rare\n        ",
 'tags': ['cast on resolution',
  'cast trigger',
  'copy-instant',
  'copy-sorcery',
  'free-cast-another',
  'gives castable from exile',
  'storm-like',
  'symmetrical']}

In [9]:
# fine tune
import torch

from transformers import AutoTokenizer
from transformers import AutoModelForSequenceClassification
from transformers import TrainingArguments
from transformers import Trainer

import numpy as np
from sklearn.metrics import f1_score, precision_score, recall_score

class FineTuneLLMv1():
    """
    Description
    ----------
    This class contains the necessary methods to fine tune a LLM for multi-label classification

    Inputs
    ----------
    model_name = The name of the model we want to start from
    dataset = The dataset object from ScryfallDataset
    label2id = The label2id object from ScryfallDataset
    id2label = The id2label object from ScryfallDataset
    n_labels = The number of labels from the ScryfallDataset
    """
    def __init__(
        self, 
        model_name:str,
        dataset,
        label2id:dict, 
        id2label:dict,
        n_labels:int
    ):
        super().__init__()

        # store params as objects
        self.model_name = model_name
        self.dataset = dataset
        self.label2id = label2id 
        self.id2label = id2label
        self.n_labels = n_labels
        
        # initialize objects
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSequenceClassification.from_pretrained(
            model_name,
            num_labels = n_labels,
            problem_type = 'multi_label_classification',
            id2label = self.id2label,
            label2id = self.label2id
        )

        # placeholders for objects to be created later

    # === Main Methods ===

    def prepare_data(
        self,
        max_length:int = 256
    ):
        """
        Description
        ----------
        This method performs all necessary preprocessing on our dataset
        """
        # === 1. Tokenize Dataset ===
        self.dataset = self.dataset.map(
            lambda x: self.tokenizer(
                x['document'],
                truncation = True,
                padding = 'max_length',
                max_length = max_length
            )
        )

        self.dataset = self.dataset.map(
            lambda x: self._encode_labels(
                data = x,
                n_labels = self.n_labels
            )
        )

        self.dataset.set_format(
            'torch',
            columns = ['input_ids', 'attention_mask', 'labels']
        )

    def train(
        self,
        batch_size:int,
        n_epochs:int,
        learning_rate:float,
        weight_decay:float
    ):
        """
        Description
        ----------
        This method defines the training arguments and then trains the model based on
        those arguments.

        Inputs
        ----------
        batch_size = The batch size for training
        n_epochs = The number of epochs to train
        learning_rate = The learning rate for learning
        weight_decay = The weight decay rate

        Returns
        ----------

        """
        # define the training arguments
        training_args = TrainingArguments(
            output_dir = '../models/scryfall_auto_tagger',
            per_device_train_batch_size = batch_size,
            per_device_eval_batch_size = batch_size,
            num_train_epochs = n_epochs,
            eval_strategy  = 'epoch',
            save_strategy = 'epoch',
            load_best_model_at_end = True,
            metric_for_best_model = 'eval_micro_f1',
            # logging_dir = '../logs',
            learning_rate = learning_rate,
            weight_decay = weight_decay
        )

        # train the model
        trainer = Trainer(
            model = self.model,
            args = training_args,
            train_dataset = self.dataset['train'],
            eval_dataset = self.dataset['val'],
            compute_metrics = self._compute_metrics
        )
        trainer.train()

    # === Internal Methods ===

    def _encode_labels(self, data, n_labels:int):
        """
        Description
        ----------
        This method reshapes a list of scryfall tags into a multi-hot vector of labels.
        NOTE: This must be ran after we have performed initial processing of the data,
        as we need to know the total number of unique labels first.

        Inputs
        ----------
        data = A dictionary containing the information for the card we are multi-hot
            encoding
        n_labels = The number of labels in our dataset
        
        Returns
        ----------
        A transformed version of data with the tags encoded
        """

        label_vector = torch.zeros(n_labels)
        for tag in data['tags']:
            label_vector[self.label2id[tag]] = 1
        data['labels'] = label_vector

        return data
    
    def _compute_metrics(self, eval_pred):
        """
        Description
        ----------
        This method dictates which metrics we will compute as part of the training loop

        Inputs
        ----------
        eval_pred = The evaluation prediction we want to compute metrics for

        Returns
        ----------
        evals = A dict containing our evaluated metrics
        """
        # define predictions
        logits, labels = eval_pred
        probs = torch.sigmoid(torch.tensor(logits))
        preds = (probs > 0.5).int().numpy()

        # define metrics
        out = {
            'micro_f1': f1_score(labels, preds, average = 'micro'),
            'macro_f1': f1_score(labels, preds, average = 'macro'),
            'micro_precision': precision_score(labels, preds, average = 'micro'),
            'micro_recall': recall_score(labels, preds, average = 'micro')
        }

        return out

labeller = FineTuneLLMv1(
    model_name = MODEL_NAME,
    dataset = sf.dataset,
    label2id = sf.label2id,
    id2label = sf.id2label,
    n_labels = len(sf.unique_tags)
)
labeller.prepare_data(
    max_length = MAX_INPUT_LENGTH
)
labeller.train(
    batch_size = BATCH_SIZE,
    n_epochs = NUM_EPOCHS,
    learning_rate = LEARNING_RATE,
    weight_decay = WEIGHT_DECAY
)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Map:   0%|          | 0/349 [00:00<?, ? examples/s]

Map:   0%|          | 0/88 [00:00<?, ? examples/s]

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

...

c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\nccru\anaconda3\envs\personal-general\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].
